In [2]:
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn

# ==============================================================================
# 1. 定义量子设备和量子线路 (QNode)
# ==============================================================================

# 使用 PennyLane 的默认 qubit 模拟器，有 1 个量子比特
n_qubits = 1
dev = qml.device("default.qubit", wires=n_qubits)

# 定义量子线路
# @qml.qnode(dev, interface='torch') 告诉 PennyLane 这个 QNode 将与 PyTorch 交互
@qml.qnode(dev, interface='torch')
def quantum_circuit(inputs, weights):
    """
    一个简单的变分量子线路。
    
    Args:
        inputs (torch.Tensor): 输入数据，将被编码到量子态中。
        weights (torch.Tensor): 线路的可训练参数。
    """
    # 数据编码：将输入值编码为 RZ 门的旋转角度
    qml.RZ(inputs[0], wires=0)
    
    # 变分层：使用可训练的权重参数
    qml.RY(weights[0], wires=0)
    qml.RX(weights[1], wires=0)
    
    # 返回 Z 方向的期望值，这是一个经典数字
    return qml.expval(qml.PauliZ(0))

# ==============================================================================
# 2. 定义完整的混合神经网络
# ==============================================================================

class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        
        # 量子层的权重参数
        # 我们需要告诉 PyTorch 这些是需要优化的参数
        # nn.Parameter 会自动将张量注册为模型的可训练参数
        self.q_weights = nn.Parameter(torch.randn(2))
        
        # 经典线性层
        # 量子线路返回 1 个期望值，所以输入特征是 1
        # 我们希望输出 1 个值，所以输出特征是 1
        self.classical_layer = nn.Linear(1, 1)

    def forward(self, x):
        # 前向传播过程
        
        # 1. 将输入数据传递给量子线路
        # quantum_circuit 返回一个形状为 (batch_size, 1) 的张量
        q_out = quantum_circuit(x, self.q_weights)
        
        # 2. 将量子线路的输出传递给经典线性层
        # 确保量子输出的形状是线性层期望的形状
        # q_out 的形状是 [batch_size]，需要变成 [batch_size, 1]
        q_out = q_out.unsqueeze(1) 
        
        # 3. 经典层处理
        final_output = self.classical_layer(q_out)
        
        return final_output

# ==============================================================================
# 3. 训练模型
# ==============================================================================

# 准备训练数据
# 我们想学习 y = x^2，x 在 [-1, 1] 区间
X_train = torch.linspace(-1, 1, 50).reshape(-1, 1)  # 50个样本，1个特征
y_train = X_train ** 2

# 实例化模型、损失函数和优化器
model = HybridModel()
criterion = nn.MSELoss()  # 均方误差损失
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) # 随机梯度下降

# 训练循环
epochs = 50
for epoch in range(epochs):
    # 前向传播
    y_pred = model(X_train)
    
    # 计算损失
    loss = criterion(y_pred, y_train)
    
    # 反向传播和优化
    optimizer.zero_grad() # 清空旧的梯度
    loss.backward()       # 自动计算梯度（包括量子部分的梯度！）
    optimizer.step()      # 更新所有参数（量子和经典）
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("\n训练完成!")

# ==============================================================================
# 4. 查看结果
# ==============================================================================

# 查看训练后的量子权重
print(f"训练后的量子权重: {model.q_weights.data}")

# 查看训练后的经典线性层权重和偏置
print(f"训练后的经典层权重: {model.classical_layer.weight.data}")
print(f"训练后的经典层偏置: {model.classical_layer.bias.data}")

# 进行预测
with torch.no_grad():
    predictions = model(X_train)

# 可视化结果
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(X_train.numpy(), y_train.numpy(), label='真实数据')
plt.plot(X_train.numpy(), predictions.numpy(), 'r-', label='混合模型预测')
plt.title('量子-经典混合模型学习 y = x^2')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()



RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float